In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

DATASET_PATH = (
    BASE_DIR
    / "app"
    / "datasets"
    / "BRFSS2024.csv"
)

MODEL_DIR = (
    BASE_DIR
    / "app"
    / "models"
)

MODEL_PATH = (
    MODEL_DIR
    / "risk_model.pkl"
)

print("BASE_DIR:")
print(BASE_DIR)

print("\nDATASET_PATH:")
print(DATASET_PATH)

print("\nMODEL_PATH:")
print(MODEL_PATH)

BASE_DIR:
c:\Users\hemas\Downloads\MedAssist AI\backend

DATASET_PATH:
c:\Users\hemas\Downloads\MedAssist AI\backend\app\datasets\BRFSS2024.csv

MODEL_PATH:
c:\Users\hemas\Downloads\MedAssist AI\backend\app\models\risk_model.pkl


In [3]:
print("Dataset exists:", DATASET_PATH.exists())
print("Model directory exists:", MODEL_DIR.exists())

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

print("\nBRFSS2024.csv found successfully.")

Dataset exists: True
Model directory exists: True

BRFSS2024.csv found successfully.


In [4]:
CHRONIC_DISEASE_COLS = [
    "DIABETE4",
    "CVDINFR4",
    "CVDCRHD4",
    "CVDSTRK3",
    "CHCKDNY2",
    "CHCCOPD3",
    "ASTHMA3",
]

FEATURE_COLS = [
    "_AGE80",
    "_SEX",
    "_BMI5",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "EXERANY2",
    "SMOKE100",
    "DRNKANY6",
    "DIABETE4",
    "CHCKDNY2",
    "ASTHMA3",
    "CHCCOPD3",
    "HAVARTH4",
]

print("Number of features:", len(FEATURE_COLS))

for i, feature in enumerate(FEATURE_COLS, start=1):
    print(i, feature)

Number of features: 14
1 _AGE80
2 _SEX
3 _BMI5
4 GENHLTH
5 PHYSHLTH
6 MENTHLTH
7 EXERANY2
8 SMOKE100
9 DRNKANY6
10 DIABETE4
11 CHCKDNY2
12 ASTHMA3
13 CHCCOPD3
14 HAVARTH4


In [5]:
required_columns = list(
    set(FEATURE_COLS + CHRONIC_DISEASE_COLS)
)

print(
    "Loading BRFSS2024 dataset..."
)

start_time = time.time()

df = pd.read_csv(
    DATASET_PATH,
    usecols=required_columns,
    low_memory=False
)

print(
    f"Dataset loaded in "
    f"{time.time() - start_time:.2f} seconds"
)

print("Dataset shape:", df.shape)

Loading BRFSS2024 dataset...
Dataset loaded in 4.82 seconds
Dataset shape: (457670, 17)


In [8]:
X, y = load_and_preprocess_data(
    DATASET_PATH
)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Loading BRFSS2024...
Original dataset shape: (457670, 17)
Feature shape: (457670, 14)
Target shape: (457670,)


In [9]:
print("Risk Level Distribution")
print("=" * 40)

risk_names = {
    0: "Low",
    1: "Medium",
    2: "High"
}

for class_id, class_name in risk_names.items():

    count = int((y == class_id).sum())

    percentage = (
        count / len(y)
    ) * 100

    print(
        f"{class_id} - {class_name}: "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )

Risk Level Distribution
0 - Low: 283,165 (61.87%)
1 - Medium: 93,476 (20.42%)
2 - High: 81,029 (17.70%)


In [11]:
# Age
X["_AGE80"] = np.where(
    X["_AGE80"].between(18, 99),
    X["_AGE80"],
    np.nan
)
# Sex
X["_SEX"] = np.where(
    X["_SEX"] == 1,
    1,
    np.where(
        X["_SEX"] == 2,
        0,
        np.nan
    )
)
# BMI
X["_BMI5"] = np.where(
    X["_BMI5"].between(1000, 9000),
    X["_BMI5"] / 100.0,
    np.nan
)
# General health
X["GENHLTH"] = np.where(
    X["GENHLTH"].between(1, 5),
    X["GENHLTH"],
    np.nan
)
# Physical health
X["PHYSHLTH"] = np.where(
    X["PHYSHLTH"] == 88,
    0,
    np.where(
        X["PHYSHLTH"].between(1, 30),
        X["PHYSHLTH"],
        np.nan
    )
)
# Mental health
X["MENTHLTH"] = np.where(
    X["MENTHLTH"] == 88,
    0,
    np.where(
        X["MENTHLTH"].between(1, 30),
        X["MENTHLTH"],
        np.nan
    )
)
# Binary features
binary_cols = [
    "EXERANY2",
    "SMOKE100",
    "DRNKANY6",
    "CHCKDNY2",
    "ASTHMA3",
    "CHCCOPD3",
    "HAVARTH4"
]
for col in binary_cols:
    X[col] = np.where(
        X[col] == 1,
        1,
        np.where(
            X[col] == 2,
            0,
            np.nan
        )
    )
# Diabetes
X["DIABETE4"] = np.where(
    X["DIABETE4"] == 1,
    1,
    np.where(
        X["DIABETE4"].isin([2, 3, 4]),
        0,
        np.nan
    )
)
print("Feature preprocessing completed.")

Feature preprocessing completed.


In [7]:
# Disease flags

is_diab = (
    df["DIABETE4"] == 1.0
).astype(int)

is_infar = (
    df["CVDINFR4"] == 1.0
).astype(int)

is_chd = (
    df["CVDCRHD4"] == 1.0
).astype(int)

is_stroke = (
    df["CVDSTRK3"] == 1.0
).astype(int)

is_kidney = (
    df["CHCKDNY2"] == 1.0
).astype(int)

is_copd = (
    df["CHCCOPD3"] == 1.0
).astype(int)

is_asthma = (
    df["ASTHMA3"] == 1.0
).astype(int)


# Major cardiovascular event

has_cardio_event = (
    is_infar
    | is_stroke
    | is_chd
)


# Total chronic disease count

total_chronic_count = (
    is_diab
    + is_infar
    + is_chd
    + is_stroke
    + is_kidney
    + is_copd
    + is_asthma
)


# Risk level

risk_level = np.where(
    (has_cardio_event == 1)
    | (total_chronic_count >= 2),
    2,
    np.where(
        total_chronic_count == 1,
        1,
        0
    )
)

y = pd.Series(
    risk_level,
    name="Risk_Level"
)

X = df[FEATURE_COLS].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (457670, 14)
y shape: (457670,)


In [25]:
print("Number of FEATURE_COLS:", len(FEATURE_COLS))
print("Number of X columns:", len(X.columns))

print("\nX columns:")
print(X.columns.tolist())

if len(X.columns) != 14:
    raise ValueError(
        f"Expected 14 features, got {len(X.columns)}"
    )

if list(X.columns) != FEATURE_COLS:
    raise ValueError(
        "X columns do not exactly match FEATURE_COLS."
    )

print("\n✓ Exactly 14 features confirmed.")

Number of FEATURE_COLS: 14
Number of X columns: 14

X columns:
['_AGE80', '_SEX', '_BMI5', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'EXERANY2', 'SMOKE100', 'DRNKANY6', 'DIABETE4', 'CHCKDNY2', 'ASTHMA3', 'CHCCOPD3', 'HAVARTH4']

✓ Exactly 14 features confirmed.


In [9]:
print("Risk Level Distribution")
print("=" * 45)

for class_id, class_name in [
    (0, "Low"),
    (1, "Medium"),
    (2, "High")
]:

    count = int(
        (y == class_id).sum()
    )

    percentage = (
        count / len(y)
    ) * 100

    print(
        f"{class_id} ({class_name}): "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )

Risk Level Distribution
0 (Low): 283,165 (61.87%)
1 (Medium): 93,476 (20.42%)
2 (High): 81,029 (17.70%)


In [10]:
# ---------------------------------------------------------
# 1. AGE
# ---------------------------------------------------------

X["_AGE80"] = np.where(
    X["_AGE80"].between(18, 99),
    X["_AGE80"],
    np.nan
)


# ---------------------------------------------------------
# 2. SEX
# 1 = Male
# 2 = Female
# Convert to:
# 1 = Male
# 0 = Female
# ---------------------------------------------------------

X["_SEX"] = np.where(
    X["_SEX"] == 1,
    1,
    np.where(
        X["_SEX"] == 2,
        0,
        np.nan
    )
)


# ---------------------------------------------------------
# 3. BMI
# BRFSS stores BMI * 100
# ---------------------------------------------------------

X["_BMI5"] = np.where(
    X["_BMI5"].between(1000, 9000),
    X["_BMI5"] / 100.0,
    np.nan
)


# ---------------------------------------------------------
# 4. GENERAL HEALTH
# ---------------------------------------------------------

X["GENHLTH"] = np.where(
    X["GENHLTH"].between(1, 5),
    X["GENHLTH"],
    np.nan
)


# ---------------------------------------------------------
# 5. PHYSICAL HEALTH
# 88 means 0 days
# ---------------------------------------------------------

X["PHYSHLTH"] = np.where(
    X["PHYSHLTH"] == 88,
    0,
    np.where(
        X["PHYSHLTH"].between(1, 30),
        X["PHYSHLTH"],
        np.nan
    )
)


# ---------------------------------------------------------
# 6. MENTAL HEALTH
# 88 means 0 days
# ---------------------------------------------------------

X["MENTHLTH"] = np.where(
    X["MENTHLTH"] == 88,
    0,
    np.where(
        X["MENTHLTH"].between(1, 30),
        X["MENTHLTH"],
        np.nan
    )
)


# ---------------------------------------------------------
# 7. BINARY FEATURES
# ---------------------------------------------------------

binary_cols = [
    "EXERANY2",
    "SMOKE100",
    "DRNKANY6",
    "CHCKDNY2",
    "ASTHMA3",
    "CHCCOPD3",
    "HAVARTH4"
]

for col in binary_cols:

    X[col] = np.where(
        X[col] == 1,
        1,
        np.where(
            X[col] == 2,
            0,
            np.nan
        )
    )


# ---------------------------------------------------------
# 8. DIABETES
# ---------------------------------------------------------

X["DIABETE4"] = np.where(
    X["DIABETE4"] == 1,
    1,
    np.where(
        X["DIABETE4"].isin([2, 3, 4]),
        0,
        np.nan
    )
)

print("Feature preprocessing completed.")

Feature preprocessing completed.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (366136, 14)
X_test : (91534, 14)
y_train: (366136,)
y_test : (91534,)


In [12]:
imputer = SimpleImputer(
    strategy="median"
)

X_train_imp = imputer.fit_transform(
    X_train
)

X_test_imp = imputer.transform(
    X_test
)

print(
    "Training matrix:",
    X_train_imp.shape
)

print(
    "Testing matrix:",
    X_test_imp.shape
)

Training matrix: (366136, 14)
Testing matrix: (91534, 14)


In [13]:
print("X_train columns:", len(X_train.columns))
print("X_train matrix:", X_train_imp.shape)
print("FEATURE_COLS:", len(FEATURE_COLS))

if X_train_imp.shape[1] != len(FEATURE_COLS):
    raise ValueError(
        "Feature count mismatch before XGBoost training!"
    )

print("\n✓ Feature count is correct: 14")

X_train columns: 14
X_train matrix: (366136, 14)
FEATURE_COLS: 14

✓ Feature count is correct: 14


In [14]:
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("=" * 60)
print("TRAINING XGBOOST")
print("=" * 60)

start_time = time.time()

xgb_model.fit(
    X_train_imp,
    y_train
)

training_time = time.time() - start_time

print(
    f"Training completed in "
    f"{training_time:.2f} seconds"
)

TRAINING XGBOOST
Training completed in 3.85 seconds


In [15]:
print(
    "Model feature count:",
    xgb_model.n_features_in_
)

print(
    "Training feature count:",
    X_train_imp.shape[1]
)

if xgb_model.n_features_in_ != 14:
    raise ValueError(
        "XGBoost was not trained with 14 features!"
    )

print("\n✓ XGBoost is trained with exactly 14 features.")

Model feature count: 14
Training feature count: 14

✓ XGBoost is trained with exactly 14 features.


In [16]:
y_pred = xgb_model.predict(
    X_test_imp
)

print("Predictions generated.")
print("Number of predictions:", len(y_pred))

Predictions generated.
Number of predictions: 91534


In [17]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("XGBOOST RESULTS")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"Macro F1 : {f1:.4f}"
)

XGBOOST RESULTS
Accuracy : 0.9091 (90.91%)
Precision: 0.9112
Recall   : 0.8302
Macro F1 : 0.8441


In [18]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Low",
            "Medium",
            "High"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         Low       0.93      1.00      0.96     56633
      Medium       0.84      0.99      0.91     18695
        High       0.97      0.50      0.66     16206

    accuracy                           0.91     91534
   macro avg       0.91      0.83      0.84     91534
weighted avg       0.92      0.91      0.90     91534



In [19]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix")
print("=" * 40)
print(cm)

Confusion Matrix
[[56581     0    52]
 [    0 18461   234]
 [ 4585  3452  8169]]


In [20]:
print(
    "Model features:",
    len(xgb_model.feature_importances_)
)

print(
    "Training features:",
    len(X_train.columns)
)

importance = pd.DataFrame({
    "Feature": X_train.columns.tolist(),
    "Importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

display(importance)

Model features: 14
Training features: 14


,Feature,Importance
0,DIABETE4,0.371561
1,ASTHMA3,0.236249
2,CHCCOPD3,0.226489
3,CHCKDNY2,0.115177
4,GENHLTH,0.014031
5,_AGE80,0.012840
6,_SEX,0.007649
7,HAVARTH4,0.006734
8,SMOKE100,0.004708
9,PHYSHLTH,0.001585


In [21]:
X_all_imp = pd.DataFrame(
    imputer.transform(X),
    columns=X.columns
)

X_all_imp["Risk_Level"] = y.values

cleaned_path = (
    BASE_DIR
    / "app"
    / "datasets"
    / "BRFSS2024_Cleaned.csv"
)

X_all_imp.to_csv(
    cleaned_path,
    index=False
)

print(
    "Cleaned dataset saved to:"
)

print(cleaned_path)

Cleaned dataset saved to:
c:\Users\hemas\Downloads\MedAssist AI\backend\app\datasets\BRFSS2024_Cleaned.csv


In [22]:
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_payload = {
    "model": xgb_model,

    "imputer": imputer,

    "model_name": "XGBoost Classifier",

    "feature_cols": FEATURE_COLS,

    "target_labels": {
        0: "Low",
        1: "Medium",
        2: "High"
    },

    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    }
}

joblib.dump(
    model_payload,
    MODEL_PATH
)

print("=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)

print("Path:")
print(MODEL_PATH)

MODEL SAVED SUCCESSFULLY
Path:
c:\Users\hemas\Downloads\MedAssist AI\backend\app\models\risk_model.pkl


In [23]:
loaded_artifact = joblib.load(
    MODEL_PATH
)

print("Artifact loaded successfully.")

print(
    "Model:",
    loaded_artifact["model_name"]
)

print(
    "Number of features:",
    len(loaded_artifact["feature_cols"])
)

print(
    "Features:",
    loaded_artifact["feature_cols"]
)

print(
    "Metrics:",
    loaded_artifact["metrics"]
)

Artifact loaded successfully.
Model: XGBoost Classifier
Number of features: 14
Features: ['_AGE80', '_SEX', '_BMI5', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'EXERANY2', 'SMOKE100', 'DRNKANY6', 'DIABETE4', 'CHCKDNY2', 'ASTHMA3', 'CHCCOPD3', 'HAVARTH4']
Metrics: {'accuracy': 0.9090720388052527, 'precision': 0.9112272859858225, 'recall': 0.8302125524812606, 'f1_score': 0.8441231982378316}


In [27]:
sample_patient = pd.DataFrame([{
    "_AGE80": 55,
    "_SEX": 1,
    "_BMI5": 29.5,
    "GENHLTH": 4,
    "PHYSHLTH": 10,
    "MENTHLTH": 5,
    "EXERANY2": 0,
    "SMOKE100": 1,
    "DRNKANY6": 1,
    "DIABETE4": 1,
    "CHCKDNY2": 0,
    "ASTHMA3": 0,
    "CHCCOPD3": 0,
    "HAVARTH4": 1
}])

sample_imputed = loaded_artifact[
    "imputer"
].transform(sample_patient)

loaded_model = loaded_artifact[
    "model"
]

predicted_class = int(
    loaded_model.predict(
        sample_imputed
    )[0]
)

probabilities = loaded_model.predict_proba(
    sample_imputed
)[0]

target_labels = loaded_artifact[
    "target_labels"
]

predicted_risk = target_labels[
    predicted_class
]

risk_probability = float(
    probabilities[predicted_class]
)

print("=" * 50)
print("SAMPLE PATIENT")
print("=" * 50)

print(
    "Risk Probability:",
    round(risk_probability, 4)
)

print(
    "Predicted Risk Level:",
    predicted_risk
)

SAMPLE PATIENT
Risk Probability: 0.7028
Predicted Risk Level: Medium
